In [534]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import cohen_kappa_score
from sklearn import metrics
import matplotlib.pyplot as plt

# ============================================================
# 1. CHARGEMENT & SETUP
# ============================================================

In [535]:
def scan_eval_folder(eval_folder: str) -> dict:
    """
    Scans the eval_ontologie folder and returns the structure of annotator folders.

    Parameters
    ----------
    eval_folder : str
        Path to the root evaluation folder containing one subfolder per annotator.

    Returns
    -------
    dict : {annotator_name: {'labels_file': Path, 'labels_dir': Path}}
    """
    eval_data_folder = Path(eval_folder)
    result = {}

    for ann_folder in eval_data_folder.iterdir():
        # Skip any loose files at the root level
        if not ann_folder.is_dir():
            continue

        annotator = ann_folder.name
        # Convert generator to list so we can iterate multiple times
        contents = list(ann_folder.iterdir())

        # Find the labels mapping file and the annotations directory
        labels_file = next((f for f in contents if f.name == "labels.txt"), None)
        labels_dir = next((f for f in contents if f.is_dir()), None)

        result[annotator] = {
            'labels_file': labels_file,
            'labels_dir': labels_dir
        }

    return result

In [536]:
def get_labels(labels_file):
    """
    This functions checks if the file 'labels.txt' exists. 
    If not, it generated a .txt file with the generic names for each existing class "class1" to "classN". 
    The users can then change the names later.
    
    **Beware: if defined classes have not been used in the training dataset, they will not appear in this labels.txt file.**

    :param labels_file: 
        - Type: str
        - Description: The path to the 'labels.txt' file which contains the class IDs and corresponding class names.
    
    :return: 
        - Type: dict
        - Description: A dictionary where keys are class IDs (as strings) and values are class names.    
    """
    labels_dict = {}
    with open(labels_file, 'r') as labels:
        for line in labels:
            key, value = line.strip().split(': ')
            key = key.strip("'")
            value = value.strip("'\n")
            labels_dict[key] = value
    
    return labels_dict

In [537]:
def load_annotator_data(all_data: dict) -> dict:
    """
    Loads annotations and class mappings for all annotators.

    Parameters
    ----------
    all_data : dict
        Output of scan_eval_folder:
        {annotator_name: {'labels_file': Path, 'labels_dir': Path}}

    Returns
    -------
    dict : {annotator_name: (annotations, labels)}
        - annotations : dict {stem: [[class_id, x, y, w, h], ...]}
        - labels : dict {class_id_str: class_name}
    """
    result = {}

    for annotator, data in all_data.items():
        labels_file = data['labels_file']
        labels_dir = data['labels_dir']

        # Load class ID → class name mapping
        labels = get_labels(str(labels_file))

        annotations = {}
        for txt_file in labels_dir.iterdir():
            # Skip non-annotation files
            if txt_file.suffix != '.txt':
                continue

            stem = txt_file.stem
            boxes = []
            with open(txt_file, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts:
                        continue
                    class_id = int(parts[0])
                    coords = [float(x) for x in parts[1:]]
                    boxes.append([class_id] + coords)

            annotations[stem] = boxes

        result[annotator] = (annotations, labels)

    return result

In [538]:
def harmonize_labels(all_data: dict) -> dict:
    """
    Builds a global class mapping across all annotators.

    Since each annotator may have different numeric IDs for the same class,
    this function aligns them by class name.

    Parameters
    ----------
    all_data : dict
        {annotator_name: {'labels_file': Path, 'labels_dir': Path}}

    Returns
    -------
    dict : {class_name: {annotator_name: class_id_int}}
    """
    mapping = {}

    for annotator, data in all_data.items():
        labels = get_labels(str(data['labels_file']))

        for class_id, class_name in labels.items():
            if class_name not in mapping:
                # First annotator encountered for this class
                mapping[class_name] = {annotator: int(class_id)}
            else:
                # Add this annotator's ID for the existing class
                mapping[class_name][annotator] = int(class_id)

    return mapping


In [539]:
def get_common_files(all_data: dict) -> dict:
    """
    Returns the stems of images annotated by at least two annotators.

    Parameters
    ----------
    all_data : dict
        {annotator_name: {'labels_file': Path, 'labels_dir': Path}}

    Returns
    -------
    dict : {stem: [annotator1, annotator2, ...]}
        Only stems annotated by at least 2 annotators are included.
    """
    common_files = {}

    for annotator, data in all_data.items():
        for file in Path(data['labels_dir']).iterdir():
            stem = file.stem
            if stem not in common_files:
                # First annotator encountered for this stem
                common_files[stem] = [annotator]
            else:
                # Add this annotator to the existing list
                common_files[stem].append(annotator)

    # Collect stems annotated by only one annotator
    exclusive_stems = [
        stem for stem, annotators in common_files.items()
        if len(annotators) < 2
    ]

    # Remove exclusive stems from the result
    for stem in exclusive_stems:
        common_files.pop(stem)

    return common_files

# ============================================================
# 2. MATCHING & MÉTRIQUES
# ============================================================

In [540]:
def calculate_iou(box1:list, box2:list) -> float:
    """
    Calculates the Intersection over Union (IoU) between two bounding boxes
    in YOLO normalized format.

    Adapted from the 'bb_intersection_over_union' function on PyImageSearch.
    Coordinates are converted from (x_center, y_center, width, height) to
    (x_min, y_min, x_max, y_max) before computing the IoU.

    Parameters
    ----------
    box1 : list [class_id, x_center, y_center, width, height]
        First bounding box in YOLO normalized format.
    box2 : list [class_id, x_center, y_center, width, height]
        Second bounding box in YOLO normalized format.

    Returns
    -------
    float : IoU value between 0 and 1.
        0 indicates no overlap, 1 indicates perfect overlap.
    """
    
    # Convert coordinates (x, y, w, h) in (x_min, y_min, x_max, y_max)
    box1_x_min = box1[1] - box1[3] / 2
    box1_y_min = box1[2] - box1[4] / 2
    box1_x_max = box1[1] + box1[3] / 2
    box1_y_max = box1[2] + box1[4] / 2
    
    box2_x_min = box2[1] - box2[3] / 2
    box2_y_min = box2[2] - box2[4] / 2
    box2_x_max = box2[1] + box2[3] / 2
    box2_y_max = box2[2] + box2[4] / 2
    
    # Calculate coordinates (x,y) of the overlap
    x_min = max(box1_x_min, box2_x_min)
    y_min = max(box1_y_min, box2_y_min)
    x_max = min(box1_x_max, box2_x_max)
    y_max = min(box1_y_max, box2_y_max)
    
    # Calculate the area of the overlap
    intersection_area = max(0, x_max - x_min) * max(0, y_max - y_min)

    # Calculer the area of the two bounding boxes
    box1_area = (box1_x_max - box1_x_min) * (box1_y_max - box1_y_min)
    box2_area = (box2_x_max - box2_x_min) * (box2_y_max - box2_y_min)
    
    # Calculate the Intersection over Union (IoU)
    iou = intersection_area / float(box1_area + box2_area - intersection_area)
    
    return iou

In [541]:
def get_hungarian_matches(boxes_a: list, boxes_b: list, iou_threshold: float = 0.5) -> dict:
    """
    Finds the optimal 1-to-1 matching between two sets of bounding boxes
    using the Hungarian algorithm (scipy.optimize.linear_sum_assignment).

    Unlike greedy matching, each box can only be matched to one other box,
    which is more appropriate for inter-annotator comparison where each
    annotator should have annotated each object once.

    Parameters
    ----------
    boxes_a : list of [class_id, x, y, w, h]
    boxes_b : list of [class_id, x, y, w, h]
    iou_threshold : float
        Minimum IoU to consider a match valid (default 0.5)

    Returns
    -------
    dict : {
        'matched': [(box_a, box_b, iou), ...],
        'unmatched_a': [box_a, ...],
        'unmatched_b': [box_b, ...]
    }
    """
    # Edge case : one of the lists is empty
    if not boxes_a:
        return {'matched': [], 'unmatched_a': [], 'unmatched_b': boxes_b}
    if not boxes_b:
        return {'matched': [], 'unmatched_a': boxes_a, 'unmatched_b': []}

    # Build IoU matrix (rows = boxes_a, cols = boxes_b)
    iou_matrix = np.zeros((len(boxes_a), len(boxes_b)))
    for i, box_a in enumerate(boxes_a):
        for j, box_b in enumerate(boxes_b):
            iou_matrix[i, j] = calculate_iou(box_a, box_b)

    # Hungarian algorithm minimizes cost → we pass (1 - IoU) as cost matrix
    row_indices, col_indices = linear_sum_assignment(1 - iou_matrix)

    matched = []
    matched_a = set()
    matched_b = set()

    for i, j in zip(row_indices, col_indices):
        iou = iou_matrix[i, j]
        if iou >= iou_threshold:
            # Valid match : IoU above threshold
            matched.append((boxes_a[i], boxes_b[j], iou))
            matched_a.add(i)
            matched_b.add(j)
        # If IoU below threshold, both boxes remain unmatched

    # Collect unmatched boxes
    unmatched_a = [boxes_a[i] for i in range(len(boxes_a)) if i not in matched_a]
    unmatched_b = [boxes_b[j] for j in range(len(boxes_b)) if j not in matched_b]

    return {
        'matched': matched,
        'unmatched_a': unmatched_a,
        'unmatched_b': unmatched_b
    }

In [542]:

def compute_pair_metrics(all_data: dict, mapping: dict, common_files: dict,
                         ann_a: str, ann_b: str, iou_threshold: float = 0.5) -> dict:
    """
    Calculates all metrics for a pair of annotators on their common images.
 
    Returns
    -------
    dict : {
        'mean_iou': float,            # mean IoU over all matched boxes
        'box_ious': [float],          # IoU of every matched box (all images pooled)
        'agreement_rate': float,      # same-class boxes / ALL boxes (unmatched included)
        'kappa': float,               # Cohen's kappa, unmatched boxes = extra 'unmatched' label
        'per_class_kappa': {class_name: float},
        'per_file_iou': {stem: float},  # per-image mean IoU (0 if an image has boxes but no match): only used to list such images
        'class_counts': {class_name: (n_boxes_ann_a, n_boxes_ann_b)},
        'class_agreed': {class_name: n_boxes_labelled_with_it_by_both},
        'n_files': int,               # images annotated by both annotators
        'n_matched': int,             # matched boxes
        'n_unmatched': int            # unmatched boxes (both annotators)
    }
    """
    annotations_a, labels_a = all_data[ann_a]
    annotations_b, labels_b = all_data[ann_b]
 
    id_to_name_a = {mapping[k][ann_a]: k for k in mapping if ann_a in mapping[k]}
    id_to_name_b = {mapping[k][ann_b]: k for k in mapping if ann_b in mapping[k]}
 
    all_iou = []
    all_classes_a = []
    all_classes_b = []
    per_file_iou = {}
    n_files = 0
    n_matched = 0
    n_unmatched = 0
 
    for stem, annotators in common_files.items():
        if ann_a not in annotators or ann_b not in annotators:
            continue
        n_files += 1
 
        boxes_a = annotations_a.get(stem, [])
        boxes_b = annotations_b.get(stem, [])
 
        # Nothing drawn by either annotator: neither agreement nor disagreement
        # to measure, so the image must not count as an IoU of 0.
        if not boxes_a and not boxes_b:
            continue
 
        result = get_hungarian_matches(boxes_a, boxes_b, iou_threshold)
        n_matched += len(result['matched'])
        n_unmatched += len(result['unmatched_a']) + len(result['unmatched_b'])
 
        file_ious = []
        for box_a, box_b, iou in result['matched']:
            all_iou.append(iou)
            file_ious.append(iou)
            all_classes_a.append(id_to_name_a.get(box_a[0], 'unknown'))
            all_classes_b.append(id_to_name_b.get(box_b[0], 'unknown'))
 
        for box_a in result['unmatched_a']:
            all_classes_a.append(id_to_name_a.get(box_a[0], 'unknown'))
            all_classes_b.append('unmatched')
 
        for box_b in result['unmatched_b']:
            all_classes_a.append('unmatched')
            all_classes_b.append(id_to_name_b.get(box_b[0], 'unknown'))
 
        per_file_iou[stem] = np.mean(file_ious) if file_ious else 0.0
 
    mean_iou = np.mean(all_iou) if all_iou else 0.0
 
    agreements = sum(1 for a, b in zip(all_classes_a, all_classes_b) if a == b and a != 'unmatched')
    total = len(all_classes_a)
    agreement_rate = agreements / total if total > 0 else 0.0
 
    kappa = cohen_kappa_score(all_classes_a, all_classes_b) if len(set(all_classes_a)) > 1 else 1.0
 
    per_class_kappa = {}
    all_class_names = set(all_classes_a + all_classes_b) - {'unmatched'}
    for class_name in all_class_names:
        binary_a = [1 if c == class_name else 0 for c in all_classes_a]
        binary_b = [1 if c == class_name else 0 for c in all_classes_b]
        if len(set(binary_a)) > 1 or len(set(binary_b)) > 1:
            per_class_kappa[class_name] = cohen_kappa_score(binary_a, binary_b)
        else:
            per_class_kappa[class_name] = 1.0
 
    # How many boxes each annotator labelled with each class (matched or not)
    class_counts = {c: (all_classes_a.count(c), all_classes_b.count(c))
                    for c in all_class_names}
 
    # Boxes that both annotators labelled with this very class
    class_agreed = {c: sum(1 for a, b in zip(all_classes_a, all_classes_b) if a == b == c)
                    for c in all_class_names}
 
    return {
        'mean_iou': mean_iou,
        'agreement_rate': agreement_rate,
        'kappa': kappa,
        'per_class_kappa': per_class_kappa,
        'per_file_iou': per_file_iou,
        'box_ious': all_iou,
        'class_counts': class_counts,
        'class_agreed': class_agreed,
        'n_files': n_files,
        'n_matched': n_matched,
        'n_unmatched': n_unmatched,
    }


# ============================================================
# 3. MATRICES & VISUALISATIONS
# ============================================================

In [543]:
def build_annotator_matrix(all_pair_metrics: dict, annotators: list, metric: str) -> pd.DataFrame:
    """
    Builds a symmetric NxN matrix for a given metric across all annotator pairs.

    Parameters
    ----------
    all_pair_metrics : dict
        {(ann_a, ann_b): {'mean_iou': float, 'agreement_rate': float, 
                          'kappa': float, 'per_class_kappa': dict, 'per_file_iou': dict}}
    annotators : list
        List of annotator names, used as row and column labels.
    metric : str
        Metric to display : 'mean_iou', 'agreement_rate' or 'kappa'

    Returns
    -------
    pd.DataFrame : symmetric NxN matrix with annotator names as index and columns.
        Diagonal is set to 1.0 (perfect agreement with oneself).
    """
    # Initialize NxN matrix with zeros
    matrix = pd.DataFrame(
        np.zeros((len(annotators), len(annotators))),
        index=annotators,
        columns=annotators
    )

    # Fill diagonal with 1.0 (perfect agreement with oneself)
    for ann in annotators:
        matrix.loc[ann, ann] = 1.0

    # Fill upper and lower triangle (symmetric matrix)
    for (ann_a, ann_b), metrics in all_pair_metrics.items():
        value = metrics[metric]
        matrix.loc[ann_a, ann_b] = value
        matrix.loc[ann_b, ann_a] = value  # symmetric

    return matrix

In [544]:
def plot_annotator_matrix(matrix: pd.DataFrame, title: str, output_path: Path = None) -> None:
    """
    Displays an inter-annotator matrix as a heatmap.

    Parameters
    ----------
    matrix : pd.DataFrame
        Symmetric NxN matrix with annotator names as index and columns.
        Output of build_annotator_matrix.
    title : str
        Title of the plot.
    output_path : Path, optional
        If provided, the figure is saved to this path before display.
        If None, the figure is only displayed (default None).

    Returns
    -------
    None
    """
    fig, ax = plt.subplots(figsize=(8, 6))

    # Draw heatmap with values annotated in each cell
    im = ax.imshow(matrix.values, vmin=0, vmax=1, cmap='YlGn')

    # Add colorbar
    plt.colorbar(im, ax=ax)

    # Set axis labels
    ax.set_xticks(range(len(matrix.columns)))
    ax.set_yticks(range(len(matrix.index)))
    ax.set_xticklabels(matrix.columns, rotation=45, ha='right')
    ax.set_yticklabels(matrix.index)

    # Annotate each cell with its value
    for i in range(len(matrix.index)):
        for j in range(len(matrix.columns)):
            value = matrix.iloc[i, j]
            # Dark text on light cells, light text on dark cells
            color = 'black' if value > 0.5 else 'white'
            ax.text(j, i, f'{value:.2f}', ha='center', va='center', color=color)

    ax.set_title(title)
    plt.tight_layout()

    if output_path is None:
        plt.show()
    else:
        plt.savefig(output_path)
        plt.show()

In [545]:
def plot_iou_distribution(all_pair_metrics: dict, output_path: Path = None,
                          iou_threshold: float = 0.5) -> None:
    """
    Displays, for each annotator pair, the distribution of the IoU of ALL matched boxes
    (all common images pooled), as boxplots. The red dot is the mean, i.e. the 'Mean IoU'
    of the report. Matched boxes have IoU >= iou_threshold by construction, so unmatched
    boxes (no counterpart, or IoU below the threshold) do not appear here: see the
    matched / unmatched counts of the report.
    """
    labels = []
    values = []
 
    for (ann_a, ann_b), pair_metrics in all_pair_metrics.items():
        labels.append(f'{ann_a}\nvs\n{ann_b}')
        values.append(pair_metrics['box_ious'])
 
    fig, ax = plt.subplots(figsize=(len(labels) * 2, 6))
 
    # 'labels' was renamed 'tick_labels' in matplotlib 3.9 (and removed in 3.11)
    try:
        ax.boxplot(values, tick_labels=labels)
    except TypeError:
        ax.boxplot(values, labels=labels)
 
    for i, iou_values in enumerate(values):
        mean = np.mean(iou_values) if len(iou_values) else 0
        ax.plot(i + 1, mean, 'ro', markersize=8, label='Mean IoU' if i == 0 else '')
 
    ax.set_ylim(iou_threshold - 0.05, 1.0)
    ax.set_ylabel('IoU of matched boxes')
    ax.set_title('IoU distribution of matched boxes per annotator pair')
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.5)
 
    plt.tight_layout()
 
    if output_path is None:
        plt.show()
    else:
        plt.savefig(output_path)
        plt.show()


In [546]:
def plot_confusion_matrix_pair(all_data: dict, mapping: dict, common_files: dict,
                                ann_a: str, ann_b: str, iou_threshold: float = 0.5, output_path: Path = None) -> None:
    """
    Displays the confusion matrix of classes for a pair of annotators.

    Only matched boxes (IoU >= threshold) are considered.
    Unmatched boxes are mapped to 'unmatched' class.

    Parameters
    ----------
    all_data : dict
        {annotator_name: (annotations, labels)}
    mapping : dict
        {class_name: {annotator_name: class_id_int}}
    common_files : dict
        {stem: [annotator1, annotator2, ...]}
    ann_a : str
        Name of the first annotator.
    ann_b : str
        Name of the second annotator.
    iou_threshold : float
        Minimum IoU to consider a match valid (default 0.5)
    output_path : Path, optional
        If provided, the figure is saved to this path before display.
        If None, the figure is only displayed (default None).

    Returns
    -------
    None
    """
    annotations_a, _ = all_data[ann_a]
    annotations_b, _ = all_data[ann_b]

    # Build inverted mapping {class_id_int: class_name} for each annotator
    id_to_name_a = {mapping[k][ann_a]: k for k in mapping if ann_a in mapping[k]}
    id_to_name_b = {mapping[k][ann_b]: k for k in mapping if ann_b in mapping[k]}

    all_classes_a = []
    all_classes_b = []

    for stem, annotators in common_files.items():
        if ann_a not in annotators or ann_b not in annotators:
            continue

        boxes_a = annotations_a.get(stem, [])
        boxes_b = annotations_b.get(stem, [])

        result = get_hungarian_matches(boxes_a, boxes_b, iou_threshold)

        for box_a, box_b, iou in result['matched']:
            all_classes_a.append(id_to_name_a.get(box_a[0], 'unknown'))
            all_classes_b.append(id_to_name_b.get(box_b[0], 'unknown'))

        for box_a in result['unmatched_a']:
            all_classes_a.append(id_to_name_a.get(box_a[0], 'unknown'))
            all_classes_b.append('unmatched')

        for box_b in result['unmatched_b']:
            all_classes_a.append('unmatched')
            all_classes_b.append(id_to_name_b.get(box_b[0], 'unknown'))

    # All unique class labels
    all_labels = sorted(set(all_classes_a + all_classes_b))

    # Compute confusion matrix
    cm = metrics.confusion_matrix(all_classes_a, all_classes_b, labels=all_labels)
    cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=all_labels)

    fig, ax = plt.subplots(figsize=(10, 8))
    cm_display.plot(ax=ax, xticks_rotation=45, cmap='Blues', values_format='d')

    ax.set_title(f'Confusion matrix : {ann_a} (y) vs {ann_b} (x)')
    ax.set_ylabel(ann_a)
    ax.set_xlabel(ann_b)

    if output_path is None:
        plt.show()
    else:
        plt.savefig(output_path)
        plt.show()

# ============================================================
# 4. RAPPORT SYNTHÉTIQUE
# ============================================================

In [547]:
def get_one_sided_classes(all_pair_metrics: dict) -> pd.DataFrame:
    """
    Lists, for each pair, the classes that one of the two annotators never used.
    For such a class the per-class Kappa is always 0 and says nothing about the
    class definition: it reflects an annotation protocol issue (class not annotated,
    or named differently), not a perceptual disagreement.
 
    Returns
    -------
    pd.DataFrame : columns ['pair', 'class_name', 'boxes_first', 'boxes_second']
    """
    rows = []
    for (ann_a, ann_b), m in all_pair_metrics.items():
        for class_name, (n_a, n_b) in m['class_counts'].items():
            if min(n_a, n_b) == 0:
                rows.append({'pair': f'{ann_a} vs {ann_b}', 'class_name': class_name,
                             'boxes_first': n_a, 'boxes_second': n_b})
    return pd.DataFrame(rows)


In [548]:
def get_one_sided_classes(all_pair_metrics: dict) -> pd.DataFrame:
    """
    Lists, for each pair, the classes that one of the two annotators never used.
    For such a class the per-class Kappa is always 0 and says nothing about the
    class definition: it reflects an annotation protocol issue (class not annotated,
    or named differently), not a perceptual disagreement.
 
    Returns
    -------
    pd.DataFrame : columns ['pair', 'class_name', 'boxes_first', 'boxes_second']
    """
    rows = []
    for (ann_a, ann_b), m in all_pair_metrics.items():
        for class_name, (n_a, n_b) in m['class_counts'].items():
            if min(n_a, n_b) == 0:
                rows.append({'pair': f'{ann_a} vs {ann_b}', 'class_name': class_name,
                             'boxes_first': n_a, 'boxes_second': n_b})
    return pd.DataFrame(rows)


In [549]:

def get_problematic_classes(all_pair_metrics: dict, kappa_threshold: float = 0.6) -> pd.DataFrame:
    """
    Identifies classes with a mean Cohen Kappa below the threshold, over the pairs
    in which BOTH annotators used the class (one-sided pairs are reported separately
    by get_one_sided_classes).
 
    Returns
    -------
    pd.DataFrame : columns ['class_name', 'mean_kappa', 'min_kappa', 'nb_pairs_below_threshold']
        (empty if no class is below the threshold)
    """
    class_kappas = {}
 
    for (ann_a, ann_b), metrics in all_pair_metrics.items():
        for class_name, kappa in metrics['per_class_kappa'].items():
            n_a, n_b = metrics['class_counts'][class_name]
            if min(n_a, n_b) == 0:
                continue  # label used by one annotator only
            class_kappas.setdefault(class_name, []).append(kappa)
 
    rows = []
    for class_name, kappas in class_kappas.items():
        mean_kappa = np.mean(kappas)
        if mean_kappa < kappa_threshold:
            rows.append({
                'class_name': class_name,
                'mean_kappa': round(mean_kappa, 3),
                'min_kappa': round(min(kappas), 3),
                'nb_pairs_below_threshold': sum(1 for k in kappas if k < kappa_threshold)
            })
 
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values('mean_kappa').reset_index(drop=True)


In [550]:
def get_divergent_pairs(all_pair_metrics: dict, iou_threshold: float = 0.5) -> pd.DataFrame:
    """
    Identifies annotator pairs with a mean IoU below the threshold.
 
    Returns
    -------
    pd.DataFrame : columns ['pair', 'mean_iou', 'agreement_rate', 'kappa']
        (empty if no pair is below the threshold; the message is printed by generate_report)
    """
    rows = []
 
    for (ann_a, ann_b), metrics in all_pair_metrics.items():
        if metrics['mean_iou'] < iou_threshold:
            rows.append({
                'pair': f'{ann_a} vs {ann_b}',
                'mean_iou': round(metrics['mean_iou'], 3),
                'agreement_rate': round(metrics['agreement_rate'], 3),
                'kappa': round(metrics['kappa'], 3)
            })
 
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values('mean_iou').reset_index(drop=True)


In [551]:

def generate_report(annotators: list, all_pair_metrics: dict, output_file: str | Path,
                    iou_threshold: float = 0.5, kappa_threshold: float = 0.6) -> None:
    """
    Generates a synthetic report, printed in the notebook and saved to a .txt file.
 
    Parameters
    ----------
    annotators : list
        List of annotator names.
    all_pair_metrics : dict
        Output of compute_pair_metrics for every pair.
    output_file : str | Path
        Path of the .txt report.
    iou_threshold : float
        IoU threshold used for divergent pair detection (default 0.5)
    kappa_threshold : float
        Kappa threshold used for problematic class detection (default 0.6)
    """
    lines = []
 
    def log(text=""):
        """Print to the notebook and keep the line for the .txt report."""
        print(text)
        lines.append(str(text))
 
    log("=" * 60)
    log("ONTOLOGY ROBUSTNESS EVALUATION — SUMMARY REPORT")
    log("=" * 60)
 
    log(f"\n📋 Annotators ({len(annotators)}) : {', '.join(annotators)}")
    log(f"📊 Number of pairs evaluated : {len(all_pair_metrics)}")
 
    # Global metrics per pair
    log("\n--- Global metrics per pair ---")
    for (ann_a, ann_b), m in all_pair_metrics.items():
        log(f"\n  {ann_a} vs {ann_b}")
        log(f"    Images annotated by both  : {m['n_files']}")
        log(f"    Boxes matched / unmatched : {m['n_matched']} / {m['n_unmatched']}")
        log(f"    Mean IoU (matched boxes)  : {m['mean_iou']:.3f}")
        log(f"    Agreement rate            : {m['agreement_rate']:.3f}")
        log(f"    Cohen Kappa               : {m['kappa']:.3f}")
        zero = sorted(s for s, v in m['per_file_iou'].items() if v == 0.0)
        if zero:
            log(f"    Images with boxes but no match (IoU = 0): {len(zero)} -> {', '.join(zero)}")
 
    # Per-class Kappa, all classes
    log("\n--- Per-class Kappa (all classes) ---")
    for (ann_a, ann_b), m in all_pair_metrics.items():
        log(f"\n  {ann_a} vs {ann_b}")
        for cls, k in sorted(m['per_class_kappa'].items()):
            n_a, n_b = m['class_counts'][cls]
            agreed = m['class_agreed'][cls]
            log(f"    {cls:<28} kappa = {k:.3f}   (boxes: {n_a} vs {n_b}, same label: {agreed})")
 
    # Classes used by only one annotator
    log("\n--- Classes used by only one annotator of a pair (Kappa not interpretable) ---")
    one_sided = get_one_sided_classes(all_pair_metrics)
    if not one_sided.empty:
        log(one_sided.to_string(index=False))
    else:
        log("✅ None")
 
    # Divergent pairs
    log("\n--- Divergent pairs ---")
    divergent = get_divergent_pairs(all_pair_metrics, iou_threshold)
    if not divergent.empty:
        log(divergent.to_string(index=False))
    else:
        log(f"✅ No divergent pairs found (IoU threshold = {iou_threshold})")
 
    # Problematic classes
    log("\n--- Problematic classes (class used by both annotators, mean Kappa below threshold) ---")
    problematic = get_problematic_classes(all_pair_metrics, kappa_threshold)
    if not problematic.empty:
        log(problematic.to_string(index=False))
    else:
        log(f"✅ No problematic classes found (Kappa threshold = {kappa_threshold})")
 
    # Overall assessment
    mean_kappa_global = np.mean([m['kappa'] for m in all_pair_metrics.values()])
    mean_iou_global = np.mean([m['mean_iou'] for m in all_pair_metrics.values()])
 
    log("\n--- Overall assessment ---")
    log(f"  Mean IoU across all pairs   : {mean_iou_global:.3f}")
    log(f"  Mean Kappa across all pairs : {mean_kappa_global:.3f}")
 
    if mean_kappa_global >= 0.8:
        log("\n  ✅ Almost perfect agreement — categories were applied consistently.")
    elif mean_kappa_global >= 0.6:
        log("\n  🟡 Substantial agreement — some classes may need clarification.")
    elif mean_kappa_global >= 0.4:
        log("\n  🟠 Moderate agreement — ontology review recommended.")
    else:
        log("\n  🔴 Low agreement — ontology requires significant revision.")
 
    log("\n" + "=" * 60)
 
    Path(output_file).write_text("\n".join(lines), encoding="utf-8")
    print(f"\nReport saved to {output_file}")


# ============================================================
# 5. PIPELINE PRINCIPAL
# ============================================================

In [552]:
def run_eval_ontologie(eval_folder: str, output_folder: str = None, iou_threshold: float = 0.5, kappa_threshold: float = 0.6) -> None:
    """
    Full pipeline : scan → load → harmonize → metrics → visualizations → report.
 
    Parameters
    ----------
    eval_folder : str
        Path to the root evaluation folder containing one subfolder per annotator.
    output_folder : str, optional
        Path to the folder where figures and the .txt report will be saved.
        If None, figures are only displayed and the report is written to the
        current directory (default None).
    iou_threshold : float
        Minimum IoU to consider a match valid (default 0.5)
    kappa_threshold : float
        Kappa below which a class is considered problematic (default 0.6)
    """
    annotators_dict = scan_eval_folder(eval_folder)
    all_data = load_annotator_data(annotators_dict)
    mapping = harmonize_labels(annotators_dict)
    common_files = get_common_files(annotators_dict)
 
    annotators = list(all_data.keys())
    all_pair_metrics = {}
    for ann_a, ann_b in combinations(annotators, 2):
        all_pair_metrics[(ann_a, ann_b)] = compute_pair_metrics(
            all_data, mapping, common_files, ann_a, ann_b, iou_threshold
        )
 
    iou_matrix = build_annotator_matrix(all_pair_metrics, annotators, 'mean_iou')
    kappa_matrix = build_annotator_matrix(all_pair_metrics, annotators, 'kappa')
 
    if output_folder is not None:
        output_folder = Path(output_folder)
        output_folder.mkdir(exist_ok=True)
 
    plot_annotator_matrix(iou_matrix, title='Mean IoU between annotators',
                          output_path=output_folder / 'mean_iou_matrix.png' if output_folder else None)
 
    plot_annotator_matrix(kappa_matrix, title='Cohen Kappa between annotators',
                          output_path=output_folder / 'kappa_matrix.png' if output_folder else None)
 
    plot_iou_distribution(all_pair_metrics,
                          output_path=output_folder / 'iou_distribution.png' if output_folder else None,
                          iou_threshold=iou_threshold)
 
    for ann_a, ann_b in combinations(annotators, 2):
        plot_confusion_matrix_pair(
            all_data, mapping, common_files, ann_a, ann_b, iou_threshold,
            output_path=output_folder / f'confusion_{ann_a}_vs_{ann_b}.png' if output_folder else None
        )
 
    # 11. Generate report (CHANGED: output_file is now required, and comes before the thresholds)
    report_file = (output_folder / 'summary_report.txt') if output_folder is not None else Path('summary_report.txt')
    generate_report(annotators, all_pair_metrics, report_file, iou_threshold, kappa_threshold)


In [ ]:
run_eval_ontologie('', #YOURDATAFOLDER
                    output_folder='', #YOUROUTPUTFOLDER
                    iou_threshold = 0.5, 
                    kappa_threshold = 0.6)